# 11. Synthesis: From Lab Results to Customer Conversations

**Format:** Facilitated discussion. No code.

**Purpose:** Translate the full lab arc into language the field can use in front of a customer next week.

## 11.1 The Full Arc

Before we talk about customers, take a moment to name what we actually built.

We started with a model that had never seen the customer's documents. It answered questions fluently and confidently. It was often wrong. That was not a model failure. That was the expected behavior of a capable general-purpose model operating outside its domain.

We did not respond to that by swapping the model. We responded by building a chain of evidence, one step at a time, asking what it would take to make the model useful for this specific problem.

In Section 01, we established a baseline. We saw what the model could and could not do without any customer data. Then we ingested documents using Docling in Section 02 and turned messy PDFs into clean, structured Markdown. That gave us a reliable data layer. In Section 03, we chunked that data, embedded it, and used retrieval to feed the model only the relevant portions. That gave us grounded answers. In Section 05, we evaluated the results honestly and identified failures by category, not by gut feeling.

At each step, we measured. We compared. We looked at what changed and why.

When retrieval alone was not enough, we did not immediately reach for model adaptation. In Section 07, we tried inference-time scaling first, letting the model take multiple attempts and selecting the best answer. Only after understanding the limits of that approach did we move further. In Section 08, we generated synthetic training data from the customer's own documents. In Section 09, we adapted the model. In Section 10, we evaluated again, comparing the adapted model against the baseline and the RAG pipeline across multiple architectures.

That sequence is the shape of real engagement work. The order matters because each step produces evidence that makes the next step cheaper, faster, and less likely to fail.

The question in front of us now is not "should we fine-tune?" The question is: "Given everything we've already done, is the model the thing that's failing?" If your retrieval is working, your chunks are coherent, your evaluation shows consistent results, and the model is still producing wrong or shallow answers, then yes, the model might be the bottleneck. But if you haven't confirmed all of those things, you don't actually know where the problem lives. And changing the model won't help you find out.

What you have now is a system you can reason about. That is the prerequisite for everything that comes next.

> **Facilitator note**: Resist the urge to summarize the technical steps here. The participants just lived through them. The goal is to help them see the shape of the work, not replay the details.

## 11.2 The Ladder Is Doctrine, Not Dogma

The escalation sequence in this lab reflects how most engagements should unfold. Baseline first. Ingestion next. Retrieval before adaptation. Evaluation before commitment. That order exists for good reasons: each step produces evidence that makes the next step cheaper, faster, and less likely to fail.

But doctrine is not the same as dogma. Doctrine tells you how to think. Dogma tells you what to do regardless of what you are seeing.

There are situations where the evidence tells you to start further up the ladder. A use case where retrieval is a known architectural non-starter. A domain so specialized that the base model's behavior gaps are obvious before you run a single query. A customer who has already done the retrieval work and brought you in specifically because it is not enough. In those cases, insisting on climbing every rung in sequence is not discipline. It is theater.

What the ladder requires is not that every step be completed. It requires that you can account for the steps you skipped and explain, with evidence, why the situation justified it. "We skipped straight to fine-tuning because the customer had already proven retrieval was insufficient" is a defensible position. "We skipped straight to fine-tuning because it sounded like the right call" is not.

Use the framework to structure your thinking, not to replace it.

## 11.3 When the Model Isn't Enough

Not every failure is a model failure. Most aren't.

When an answer is wrong, the instinct is to blame the model. But in a RAG system, there are at least four places the failure could originate: ingestion could have lost structure, chunking could have fragmented meaning, retrieval could have returned the wrong context, or the prompt could have been ambiguous. The model is the last link in the chain, not the only one.

Here is how you tell the difference.

If the same question produces different answers on different runs, that is almost certainly a retrieval issue. The model is being handed a different context each time, so of course the output varies. Fix retrieval. Don't retrain the model.

If the model gets the right context but still produces a shallow or incorrect answer, that's closer to a model limitation. But even then, check the prompt first. A vague system instruction or a missing constraint can cause a capable model to underperform in ways that look like a knowledge gap.

The signal that the model genuinely isn't enough looks like this: retrieval consistently returns the right chunks, the prompt is clear and well-constrained, and the model still fails in predictable, explainable ways. The failures are not random. They cluster around specific types of reasoning: implicit rules, domain-specific terminology that the model never encountered during pretraining, or multi-step logic that requires understanding conventions the documents assume but never state.

When that pattern emerges, and you can demonstrate it with examples, you have a legitimate case for model adaptation. Not before.

The discipline here is straightforward. ***Diagnose before you prescribe***. If you can't point to exactly where and why the model is failing, you haven't earned the right to change it.

## 11.4 Evaluation-Driven Development

There's a name for what you've been doing throughout this lab, even if it hasn't been called that explicitly: Evaluation-Driven Development.

The idea is simple. Every decision to change the system, whether that's adjusting chunk size, swapping a model, rewriting a prompt, or escalating to fine-tuning, should be motivated by evaluation evidence, and validated by running that evaluation again afterward. You don't change things because they feel wrong. You change them because your evaluation set tells you something specific is failing, and you verify the fix by measuring whether it stopped.

This turns system improvement from an art into a loop. Measure. Identify the failure. Hypothesize a cause. Make one change. Measure again. If the score goes up and nothing else regressed, you've made a justified improvement. If it doesn't, you've learned something. Either way, you're not guessing.

The practical implication is that your evaluation set is not a one-time artifact. It's infrastructure. The questions you built, the expected answers, the scoring logic, all of that becomes the thing you run every time the system changes. A team that has this in place can evaluate a model swap in minutes. A team that doesn't has to re-read outputs by hand every time, which means they usually don't, which means changes accumulate without evidence.

When you talk to a customer who's stuck in a cycle of making changes and not knowing if they helped, this is the frame to offer. Before you touch the model, build the loop.

## 11.5 What the Results Actually Showed

> **Note:** The analysis below reflects the reference run included with this lab. Your live results may vary depending on training randomness and endpoint load. The patterns described are representative, not guaranteed.

The evaluation results from Section 10 are worth interpreting out loud before participants leave the room, because they contain a more nuanced story than the pass/fail counts suggest.

### Single-Model Evaluation

In our pre-built results, the summary was 6/10, then 8/10 after Best-of-N, then 8/10 after fine-tuning with RAG. On the surface, fine-tuning matched inference-time scaling and that looks like a wash.

But look at what moved and what did not.

In the reference run, Best-of-N recovered two questions through sampling. The model had the capability; it just did not surface it on the first try. Fine-tuning recovered different questions, particularly table lookups, by changing how the model reads structured information in context. The two approaches did not fix the same problems. In a production system, you would use both.

In the reference run, the model without RAG context scored 5/10, lower than the baseline of 6/10. That is not a failure of training. It is evidence that 8 examples of Thief-related content cannot teach a model about Fighters, Clerics, and Halflings. The model without context was operating on partial domain knowledge and general priors, a combination that is less reliable than either the base model or the RAG pipeline alone. The training loss itself tells the same story: it dropped from 2.93 to 2.26, well above the < 1.0 target that indicates full convergence — the model was still learning when training ended.

In the reference run, the regression on q03 is the most instructive result. The baseline answered it correctly. The fine-tuned model failed it in both evaluation modes. Fine-tuning introduced a specific wrong belief about whether spellcasters can wear armor. That is catastrophic forgetting at small scale. It is also exactly why you evaluate on a held-out test set rather than the same questions that guided your training, and why you compare against the baseline before shipping anything.

### Cross-Architecture Comparison

Section 10 expanded the evaluation to four models: the lab's Granite 8B plus three community-contributed models (Granite 2B, Phi-3 Mini, Qwen2.5 3B) all fine-tuned on the same 8 training examples with a higher learning rate (2e-4 vs 5e-6).

The results tell a clear story about where model size matters and where it does not.

Without RAG context, the models diverged. Scores ranged from 2/10 to 5/10. The larger Granite 8B led, which is expected: a larger model stores more general knowledge in its weights. Without retrieved context to ground the answer, size buys you something.

With RAG context, the models converged. All four scored 8/10 or 9/10. The smallest models matched or exceeded the 8B model. When the retriever provides the right information, what matters is the model's ability to extract the answer from the context, not how much it memorized during pretraining. Two of the smaller community models (Granite 2B, Qwen2.5 3B) actually beat the lab's 8B model by one question.

The RAG lift tells the rest of the story. Smaller models showed larger lifts (+5 to +6) compared to the 8B model (+3). That is not a weakness of smaller models. It means they benefit more from retrieval because they have less built-in knowledge to fall back on. In a deployment where RAG is always available, that is a feature, not a bug: you get comparable accuracy at lower inference cost.

The speed difference reinforces this. The smaller models were 3-10x faster at inference — the 8B model needed over 5 minutes for 10 questions without context, while the 2-3B models finished in 30-70 seconds. That gap compounds at scale: lower latency per request, higher throughput per GPU, and a meaningfully smaller infrastructure bill. When accuracy is equivalent with RAG, the speed advantage makes the smaller models hard to argue against.

The learning rate difference (40x higher for the community models) is a significant confound. A fair comparison would hold LR constant. But the practical takeaway stands: for narrow-domain SFT with minimal training data and a strong retrieval pipeline, smaller models can be a cost-effective choice.

### The Takeaway

The takeaway for the field is not "fine-tuning is risky" or "bigger is always better." It is "evaluation is what tells you whether your choices worked, and you cannot skip it." The cross-architecture results add a practical dimension: when RAG is part of the deployment, the field can confidently recommend smaller, cheaper, faster models without sacrificing answer quality.

## 11.6 The Three Questions Customers Actually Ask

In practice, enterprise AI conversations almost always reduce to three questions. The field should be able to answer all three without reaching for a slide deck.

**"Can we just use ChatGPT for this?"**

This is not a naive question. It deserves a direct answer. General-purpose models are genuinely capable for a wide range of tasks. The honest answer is: it depends on what "this" is.

If the task requires reasoning over proprietary documents that no public model has ever seen, a general-purpose model without context will hallucinate. If the task requires consistent, defensible answers that can be traced back to source material, context stuffing is fragile and expensive at scale. If the task involves domain-specific terminology that maps to concepts a general model has learned differently, you will get fluent wrong answers.

The question is not "is this model smart enough." It is "can this model be grounded in the customer's data in a way that is reliable, repeatable, and auditable." That is a different question, and it has a different answer.

**"Why isn't RAG enough?"**

RAG is enough for a lot of problems. The honest answer is that it depends on what kind of failures you are seeing.

If answers are wrong because the retriever pulled the wrong chunks, that is a retrieval problem. Better chunking, better embeddings, better retrieval strategy. RAG fixes that.

If answers are wrong because the retrieved context contains the right information and the model still cannot produce the right answer, that is a different failure. The model is receiving everything it needs and something in its prior training is interfering. That is where retrieval stops being sufficient.

The distinction matters because the fix is completely different. You cannot fine-tune your way out of a retrieval failure, and you cannot chunk your way out of a model reasoning failure. Evaluation is the only way to know which problem you have.

**"How long will this take?"**

The honest answer is that it depends on where you start and what success looks like.

A RAG pipeline on clean documents can show meaningful results in days. Evaluation and iteration adds time, but that time is not wasted, it is the work that makes the system defensible. Model adaptation is measured in weeks, not days, when you include data generation, training, evaluation, and validation against held-out questions.

The more useful question to ask the customer is: what does "good enough" look like, and who has to trust the answer. If the answer is legal or compliance, the bar is higher and the timeline reflects that. If the answer is an internal productivity tool, the bar may be lower and you can ship faster.

## 11.7 The Escalation Ladder as a Customer Conversation

The escalation ladder is not just an engineering framework. It is a sales and trust-building tool.

When a customer comes in asking for fine-tuning, the instinct is to either agree immediately or push back with a list of prerequisites. Neither approach builds trust. The escalation ladder gives you a third option: a structured conversation about what the evidence says.

The conversation sounds like this:

"Before we talk about changing the model, we want to understand what the model is doing with your data today. That tells us whether the problem is in the data, in the retrieval, or in the model itself. If it turns out to be the model, we have a clear path. If it turns out to be something else, we save you the cost and time of a training run that would not have fixed it."

That framing does three things. It positions the field as rigorous rather than reactive. It gives the customer a concrete deliverable at each stage rather than a promise at the end. And it reduces the risk of the engagement failing for the wrong reasons.

> **Facilitator note**: Ask the room: "What does a customer hear when you say this?" The answer should be "they hear that you have a process and that you are not guessing." If participants are uncertain about how to deliver this message, spend five minutes on the wording. This is the part of the lab that directly affects revenue.

## 11.8 This Works in Production

The approach demonstrated in this lab is not theoretical.

ATOSS, a German workforce management software company, faced a version of this problem with their support and documentation systems. Their documents were complex, domain-specific, and built up over years. General-purpose models answered questions fluently but not reliably. They needed answers that were grounded, consistent, and traceable.

They worked through the same escalation the field just practiced. Document ingestion. Structured retrieval. Evaluation against real customer questions. Synthetic data generation from their own documentation. Model adaptation. A three-way evaluation comparing the base model, the RAG pipeline, and the adapted model.

The adapted model improved meaningfully on the failure categories that RAG could not address. The pipeline was repeatable and survived model updates without requiring a full retraining cycle. The answers were traceable to source material, which satisfied their internal trust and compliance requirements.

That is the story the field can tell. Not "we have a fine-tuning product." Rather: "we have a process that starts with your documents, produces defensible answers, and tells you when and whether model adaptation is justified."

> **Facilitator note**: If you have access to additional customer case studies from your own engagements, this is the right place to include them. The ATOSS example establishes that the approach works at production scale. Local examples establish that the field has already done this work.

## 11.9 What to Bring Into the Next Customer Meeting

This section is about what to ask, what to say, and how to frame the work you just learned.

### What to Ask

Bring one failing question. Not a hypothetical. A real question the customer's system gets wrong today, with the wrong answer on record. That question is the starting point of the conversation about where the failure lives and what it would take to fix it.

Bring the escalation ladder, not as a slide but as a mental model. When the customer describes their problem, you are already mapping it: is this a data quality issue, a retrieval issue, or a model issue? That mapping shapes everything that follows.

Bring a timeline that separates stages. RAG pipeline with evaluation: weeks. Model adaptation with proper data generation and validation: months. Being clear about what each stage costs and what it delivers prevents the engagement from stalling when the customer realizes fine-tuning takes longer than they expected.

Do not bring a promise that you will solve it. Bring a process that will tell you whether it can be solved and what solving it requires. That is a more credible offer, and it is the one that ages well.

### What to Say

There is a mindset shift worth naming explicitly: GenAI evaluation is not a true/false test. It is a range metric. There is no state where a system is simply "working" or "not working." There is a range of performance, and the real question is whether the system is operating within an acceptable range for the specific use case, risk tolerance, and user expectations of the customer in front of you. Evaluation-Driven Development is how you find that range, track it, and defend it. Every change you make to the system should be motivated by moving a metric closer to the acceptable range, and validated by measuring whether it did.

With that frame in place, the customer conversations become cleaner.

When a customer says "We need a custom model," the instinct might be to agree and start scoping a fine-tuning engagement. Resist that instinct. Not because the customer is wrong, but because agreeing too early skips the diagnostic work that protects both of you.

Instead of "We'll generate synthetic data and fine-tune," say: "We use synthetic data to test whether model adaptation will actually help before committing to it. It lets us reduce risk, not skip steps."

Instead of "Fine-tuning will fix that," say: "Let's first confirm where the system is breaking. If the model is the bottleneck, we'll know exactly what to train on and why."

Instead of "RAG didn't work, so we need to go further," say: "RAG is working. The question is whether it's working well enough, and if not, whether the gap is in retrieval, in the data, or in the model itself."

These are not hedges. They are precise, honest statements that position you as someone who builds systems carefully rather than someone who sells techniques. The underlying message in every one of these responses is the same: we define what acceptable looks like, we measure against it, and we only escalate when the evidence says we haven't reached it yet.

## 11.10 Closing

You now have a complete pipeline from document ingestion to model adaptation, with evaluation at every stage. More importantly, you have the judgment to know when each stage is necessary and when it is not.

This lab walked through every rung of the escalation ladder. Section 01 established a baseline. Sections 02 through 04 built the retrieval pipeline. Sections 05 and 06 introduced honest evaluation. Section 07 explored inference-time scaling. Section 08 generated synthetic training data. Section 09 adapted the model. Section 10 evaluated the results across architectures and deployment configurations.

At no point did we skip to the answer. At every point, we earned the right to take the next step by proving the previous one was not sufficient.

That discipline is the actual deliverable. The models will change. The tools will change. The customers will bring problems you have not seen before. But the process of measuring before changing, diagnosing before prescribing, and evaluating before shipping does not go out of date.

The field that leaves this room is not the field that says yes to every fine-tuning request, and it is not the field that says RAG is always enough. It is the field that asks what the evidence shows and knows what to do with the answer.

That is what enterprise customers trust. That is what keeps engagements from failing for the wrong reasons. And that is what this lab was actually teaching you.